In [1]:
# ================================================================
# 📚 PDF CHATBOT USING LANGCHAIN + RAG
# Google Colab - EVERYTHING IN ONE CELL
# ================================================================

# -------------------- 1. INSTALL PACKAGES -----------------------

import sys
import subprocess

packages = [
    "langchain",
    "langchain-community",
    "langchain-huggingface",
    "langchain-text-splitters",
    "pypdf",
    "sentence-transformers",
    "faiss-cpu",
    "transformers",
    "accelerate"
]

print("Installing required packages...")

subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-q"] + packages
)

# -------------------- 2. IMPORTS -------------------------------

import os
import torch

from google.colab import files

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    pipeline
)

from langchain_huggingface import HuggingFacePipeline

# -------------------- 3. TITLE -------------------------------

print("\n" + "=" * 70)
print("             📚 LANGCHAIN PDF CHATBOT")
print("=" * 70)

print("""
This chatbot uses:

✓ LangChain
✓ RAG
✓ FAISS Vector Database
✓ Hugging Face Embeddings
✓ Hugging Face LLM
✓ PDF document retrieval
✓ Source/page references
""")

# -------------------- 4. UPLOAD PDFS ----------------------------

print("📁 Upload your PDF files below.")
print("You can upload one or multiple PDFs.\n")

uploaded = files.upload()

pdf_files = [
    filename
    for filename in uploaded.keys()
    if filename.lower().endswith(".pdf")
]

if len(pdf_files) == 0:
    raise Exception("❌ No PDF files uploaded.")

print("\nUploaded PDFs:")

for pdf in pdf_files:
    print("✓", pdf)

# -------------------- 5. LOAD PDFs ------------------------------

print("\n📖 Loading PDF documents...")

all_documents = []

for pdf_file in pdf_files:

    loader = PyPDFLoader(pdf_file)

    documents = loader.load()

    # Add filename metadata
    for document in documents:
        document.metadata["file_name"] = pdf_file

    all_documents.extend(documents)

print(
    f"✓ Loaded {len(all_documents)} pages "
    f"from {len(pdf_files)} PDF(s)."
)

# -------------------- 6. SPLIT DOCUMENTS ------------------------

print("\n✂️ Splitting documents into chunks...")

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150,
    separators=[
        "\n\n",
        "\n",
        ". ",
        " ",
        ""
    ]
)

chunks = text_splitter.split_documents(
    all_documents
)

print(f"✓ Created {len(chunks)} chunks.")

# -------------------- 7. EMBEDDING MODEL ------------------------

print("\n🧠 Loading embedding model...")

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={
        "device": "cuda"
        if torch.cuda.is_available()
        else "cpu"
    },
    encode_kwargs={
        "normalize_embeddings": True
    }
)

print("✓ Embedding model loaded.")

# -------------------- 8. CREATE VECTOR DATABASE -----------------

print("\n🔎 Creating FAISS vector database...")

vector_db = FAISS.from_documents(
    chunks,
    embedding_model
)

print(
    f"✓ Vector database created with "
    f"{len(chunks)} document chunks."
)

# -------------------- 9. LOAD LLM -------------------------------

print("\n🤖 Loading language model...")

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=(
        torch.float16
        if torch.cuda.is_available()
        else torch.float32
    ),
    device_map="auto"
)

text_generation_pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,

    max_new_tokens=400,

    do_sample=True,
    temperature=0.2,
    top_p=0.9,

    repetition_penalty=1.1
)

llm = HuggingFacePipeline(
    pipeline=text_generation_pipeline
)

print("✓ Language model loaded.")

# -------------------- 10. CREATE PROMPT -------------------------

prompt_template = """
You are a helpful PDF Question Answering Assistant.

Answer the user's question using ONLY the information
contained in the provided context.

IMPORTANT RULES:

1. Do not make up information.
2. Do not use outside knowledge.
3. If the answer is not present in the context, say:
   "I could not find the answer in the uploaded PDF."
4. Explain the answer clearly.
5. Use bullet points when useful.
6. Keep the answer appropriate for a college student.

CONTEXT:
{context}

QUESTION:
{question}

ANSWER:
"""

prompt = PromptTemplate(
    template=prompt_template,
    input_variables=[
        "context",
        "question"
    ]
)

# -------------------- 11. RETRIEVER -----------------------------

retriever = vector_db.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 5
    }
)

# -------------------- 12. ANSWER FUNCTION ----------------------

def ask_pdf(question):

    # Retrieve relevant chunks
    retrieved_docs = retriever.invoke(question)

    # Create context
    context_parts = []

    for i, doc in enumerate(
        retrieved_docs,
        start=1
    ):

        file_name = doc.metadata.get(
            "file_name",
            "Unknown PDF"
        )

        page_number = doc.metadata.get(
            "page",
            0
        )

        # PyPDFLoader uses zero-based page numbers
        page_number = page_number + 1

        context_parts.append(
            f"""
SOURCE {i}
File: {file_name}
Page: {page_number}

{doc.page_content}
"""
        )

    context = "\n".join(context_parts)

    # Create final prompt
    final_prompt = prompt.format(
        context=context,
        question=question
    )

    # Generate answer
    response = llm.invoke(
        final_prompt
    )

    # Sometimes the model returns the prompt too.
    # Try to extract only the answer.
    answer = response

    if "ANSWER:" in answer:
        answer = answer.split(
            "ANSWER:",
            1
        )[1].strip()

    return answer, retrieved_docs


# -------------------- 13. CHATBOT -------------------------------

print("\n" + "=" * 70)
print("          🎓 PDF CHATBOT IS READY!")
print("=" * 70)

print("""
Ask questions about your uploaded PDFs.

Examples:

• What is the main topic of this document?
• Explain normalization.
• What are the advantages of TCP?
• Summarize Chapter 2.
• Explain this concept in simple terms.
• What are the important points from this chapter?

Type 'exit' to stop the chatbot.
""")

# -------------------- 14. CHAT LOOP ------------------------------

while True:

    question = input("\n🧑 You: ").strip()

    if question.lower() in [
        "exit",
        "quit",
        "q"
    ]:

        print(
            "\n👋 Thank you for using "
            "the LangChain PDF Chatbot!"
        )

        break

    if not question:
        continue

    print("\n🔍 Searching PDF...")

    try:

        answer, retrieved_docs = ask_pdf(
            question
        )

        print("\n🤖 Assistant:")
        print("-" * 70)

        print(answer)

        # ---------------- SOURCES ----------------

        print("\n📚 Sources:")
        print("-" * 70)

        seen_sources = set()

        for doc in retrieved_docs:

            file_name = doc.metadata.get(
                "file_name",
                "Unknown PDF"
            )

            page = doc.metadata.get(
                "page",
                0
            ) + 1

            source = (
                file_name,
                page
            )

            if source not in seen_sources:

                print(
                    f"📄 {file_name} "
                    f"| Page {page}"
                )

                seen_sources.add(source)

        print("-" * 70)

    except Exception as error:

        print(
            "\n❌ Error occurred:"
        )

        print(error)

Installing required packages...


/tmp/ipykernel_2058/508261554.py:36: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader



             📚 LANGCHAIN PDF CHATBOT

This chatbot uses:

✓ LangChain
✓ RAG
✓ FAISS Vector Database
✓ Hugging Face Embeddings
✓ Hugging Face LLM
✓ PDF document retrieval
✓ Source/page references

📁 Upload your PDF files below.
You can upload one or multiple PDFs.



Saving BCS501-module-1-textbook.pdf to BCS501-module-1-textbook.pdf

Uploaded PDFs:
✓ BCS501-module-1-textbook.pdf

📖 Loading PDF documents...
✓ Loaded 51 pages from 1 PDF(s).

✂️ Splitting documents into chunks...
✓ Created 232 chunks.

🧠 Loading embedding model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'repetition_penalty', 'top_p', 'do_sample', 'temperature', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


✓ Language model loaded.

          🎓 PDF CHATBOT IS READY!

Ask questions about your uploaded PDFs.

Examples:

• What is the main topic of this document?
• Explain normalization.
• What are the advantages of TCP?
• Summarize Chapter 2.
• Explain this concept in simple terms.
• What are the important points from this chapter?

Type 'exit' to stop the chatbot.


🧑 You: What is the main topic of this document?

🔍 Searching PDF...


[transformers] Both `max_new_tokens` (=400) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.



🤖 Assistant:
----------------------------------------------------------------------
The main topic of this document appears to be the Unified Process, specifically its phases and generic framework activities. The passage discusses how these activities can be applied across different stages of a software process model, including the Unified Process itself. It provides details on the Unified Process' structure and how it aligns with other frameworks like the Rational Unified Process (RUP). Additionally, it mentions the importance of understanding both the initial context and the resulting context of applying a pattern within the Unified Process. However, without direct quotes from the source documents mentioned, we cannot provide exact answers about the content covered in each section. The focus seems to be on explaining the Unified Process and its relationship to various software processes.

📚 Sources:
----------------------------------------------------------------------
📄 BCS501-modu